# Phase 1: Inference & Panel Diagnostics

## The Narrative Focus
In a retail context, relying on aggregate or simple linear relationships often leads to mispriced elasticity and overestimated promotional lifts. This notebook serves as the forensic justification for our **High-Dimensional Fixed-Effects Regression Strategy**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import sys

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

# --- ROBUST PATH CONFIGURATION ---
cwd = os.getcwd()
project_root = os.path.abspath(os.path.join(cwd, ".." if "notebooks" in cwd else "."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.load_data import load_data
from src.data.preprocess import preprocess_data

transactions_path = os.path.join(project_root, 'data', 'raw', 'wcer.csv')
products_path = os.path.join(project_root, 'data', 'raw', 'upccer.csv')

transactions, products = load_data(transactions_path, products_path)
df = preprocess_data(transactions, products)
print(f"Panel Loaded: {df.shape[0]} Store-Week observations across {df['Store_ID'].nunique()} Stores.")

## 1. Data Sparsity & Completeness Check
*"Before modeling, I always assess panel balance. Are all stores open for the full 400-week observation period, or are we dealing with an unbalanced panel with temporary closures?"*

In [ ]:
store_counts = df.groupby('Store_ID')['Week'].count().sort_values(ascending=False)

plt.figure(figsize=(14, 4))
sns.histplot(store_counts, bins=30, color='#9b59b6')
plt.title('Distribution of Active Weeks per Store (Panel Balance)')
plt.xlabel('Number of Weeks with Active Sales')
plt.ylabel('Count of Stores')
plt.show()

**Insight:** The panel is highly balanced, with the vast majority of stores having ~400 weeks of history. We do not need heavy imputation for missing store histories.

## 2. Statistical Skewness: Why Log Transformations?
*"Retail data violates OLS homoscedasticity assumptions immediately. I apply a `log1p` transformation to normalize the error distribution."*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sample_df = df.sample(min(100000, len(df)), random_state=42)

sns.kdeplot(sample_df['Sales'], fill=True, ax=axes[0], color='#2E86AB')
axes[0].set_title('Raw Sales (Extreme Right-Skew)')

sns.kdeplot(np.log1p(sample_df['Sales']), fill=True, ax=axes[1], color='#E63946')
axes[1].set_title('Log(Sales) (Gaussian Approximation)')

plt.tight_layout()
plt.show()

## 3. The Endogeneity Problem: Store Heterogeneity
*"Why use High-Dimensional Fixed Effects? Because unobserved store traits (location, demographics) dictate the baseline volume. If we ignore this, our promotion lift estimates become biased by the baseline differences of the locations."*

In [ ]:
top_20_stores = df['Store_ID'].value_counts().nlargest(20).index
df_sample_stores = df[df['Store_ID'].isin(top_20_stores)]

plt.figure(figsize=(16, 6))
sns.boxplot(data=df_sample_stores, x='Store_ID', y='Sales', order=top_20_stores, palette='viridis', showfliers=False)
plt.title('Unobserved Heterogeneity: Baseline Sales Variance Across Top 20 Stores')
plt.xlabel('Store ID')
plt.ylabel('Weekly Sales')
plt.show()

## 4. The Raw Impact of Promotions
*"Let's look at the sheer unadjusted impact of a promotion on sales volume. This is the raw signal our regression engine will 'shrink' once we control for price and store contexts."*

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(data=sample_df, x='Promo', y=np.log1p(sample_df['Sales']), palette=['#95a5a6', '#2ecc71'])
plt.title('Sales Density: Baseline vs. Active Promotion')
plt.xlabel('Promotion Active (0 = No, 1 = Yes)')
plt.ylabel('Log(Sales)')
plt.show()

## 5. Price Elasticity & Confounding Factors
*"Promos are not isolated. They are deeply correlated with price cuts. The scatter plot reveals two distinct elasticity clouds, showing why we must include interaction terms (Promo $\times$ Price) in our econometrics."*

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=sample_df.sample(min(15000, len(sample_df))), 
    x=np.log1p(sample_df['Price']), 
    y=np.log1p(sample_df['Sales']), 
    hue='Promo', 
    alpha=0.3, 
    palette={0: '#34495e', 1: '#e74c3c'}
)
plt.title('Price Elasticity Regime Shift During Promotions')
plt.xlabel('Log(Price)')
plt.ylabel('Log(Sales)')
plt.show()

## 6. Structural Seasonality Analysis
*"Are sales stable year-round? The monthly boxplots help us determine if we need localized time-fixed effects (Month Dummies) alongside Store-fixed effects."*

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='Month', y=np.log1p(df['Sales']), palette='coolwarm', showfliers=False)
plt.title('Macro Monthly Seasonality (Log Variance)')
plt.xlabel('Month (1=Jan, 12=Dec)')
plt.ylabel('Log(Weekly Sales)')
plt.show()